## **Comparação de AutoML: FLAML e Auto-Sklearn com o Iris**

**Aluno:** Gustavo Pereira Prado | 11241104195

**Dataset:** [Iris Species — Kaggle](https://www.kaggle.com/datasets/menegidio/iris-species)

Neste projeto, uso o FLAML e o Auto-Sklearn para identificar as espécies das flores pelas suas medidas. Depois, comparo os acertos, o tempo, a memória e os modelos escolhidos por cada ferramenta.

## **1 - Instalação das bibliotecas**

Aqui instalo o FLAML. O Auto-Sklearn será instalado em um ambiente separado na parte 6.

In [1]:
# Instala o FLAML com as opções de AutoML.
!pip install -q "flaml[automl]"

# O Auto-Sklearn será instalado na parte 6, com Python 3.9.

## **2 - Bibliotecas e configurações**

Aqui importo as bibliotecas que vou usar e defino as configurações. Cada ferramenta terá 60 segundos para buscar os modelos, com semente 42 e um trabalhador de processamento. No Auto-Sklearn, cada tentativa pode durar até 10 segundos.

In [2]:
import os
import sys
import json
import subprocess
import time
import psutil
import numpy as np
import pandas as pd
import kagglehub

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from flaml import AutoML

# Fica True quando o Auto-Sklearn termina e salva as métricas.
AUTOSKLEARN_OK = False

# Configurações usadas na comparação.
TIME_BUDGET = 60 # Tempo de busca em segundos.
RANDOM_STATE = 42
N_JOBS = 1 # Um trabalhador para cada ferramenta.
PER_RUN_LIMIT = 10 # Tempo máximo por tentativa do Auto-Sklearn.

print(f"Python {sys.version.split()[0]} | núcleos de CPU: {os.cpu_count()} | RAM total: {psutil.virtual_memory().total / 1024**3:.1f} GB")

Python 3.13.18 | núcleos de CPU: 2 | RAM total: 12.7 GB


## **3 - Carregando o Iris**

Tento baixar o dataset pelo Kaggle. Se não der certo, uso o Iris do scikit-learn. Depois, separo as medidas das flores em `X` e a espécie em `y`. A coluna de ID, quando existe, fica de fora.

In [3]:
try:
    path = kagglehub.dataset_download("menegidio/iris-species")
    print("Arquivos na pasta:", os.listdir(path))

    # Procura um CSV nos arquivos baixados.
    csv_file = None
    for root, dirs, files in os.walk(path):
        for file in files:
            if file.endswith(".csv"):
                csv_file = os.path.join(root, file)
                break
    df = pd.read_csv(csv_file)
    print(f"Arquivo '{os.path.basename(csv_file)}' carregado com sucesso.")

except Exception as erro:
    print(f"Não foi possível usar o Kaggle ({type(erro).__name__}).")
    iris = load_iris(as_frame=True)
    df = iris.frame.copy()
    df["species"] = iris.target_names[iris.target]
    df = df.drop(columns="target")

print(df.head())
print(f"\nDimensão: {df.shape}")

# Pega a primeira coluna não numérica como alvo e separa o ID para tirar de X.
colunas_texto = df.select_dtypes(exclude="number").columns
alvo = colunas_texto[0] if len(colunas_texto) > 0 else df.columns[-1]
colunas_id = [c for c in df.columns if c.lower() == "id"]

X = df.drop(columns=[alvo] + colunas_id)
y = df[alvo]
print(f"\nAlvo: {alvo}\n{y.value_counts()}")

Using Colab cache for faster access to the 'iris-species' dataset.
Arquivos na pasta: ['Iris.csv', 'database.sqlite']
Arquivo 'Iris.csv' carregado com sucesso.
   Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0   1            5.1           3.5            1.4           0.2  Iris-setosa
1   2            4.9           3.0            1.4           0.2  Iris-setosa
2   3            4.7           3.2            1.3           0.2  Iris-setosa
3   4            4.6           3.1            1.5           0.2  Iris-setosa
4   5            5.0           3.6            1.4           0.2  Iris-setosa

Dimensão: (150, 6)

Alvo: Species
Species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64


## **4 - Separação entre treino e teste**

Uso 80% dos dados para treino e 20% para teste. O `stratify` mantém a proporção de cada espécie nos dois grupos. A semente fixa permite repetir essa mesma divisão.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE,
    stratify=y # Mantém a proporção das espécies nos dois grupos.
)
print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")

Treino: (120, 4) | Teste: (30, 4)


## **5 - FLAML**

O FLAML testa modelos dentro do tempo definido, usando a acurácia para escolher o melhor. Depois, avalio esse modelo nos dados de teste e mostro as configurações escolhidas. Também meço o tempo e a diferença de memória do processo antes e depois do treino.

In [5]:
automl_flaml = AutoML()

mem_antes = psutil.Process().memory_info().rss / 1024**2 # Memória do processo antes do treino, em MB.
t0 = time.perf_counter()

automl_flaml.fit(
    X_train, y_train,
    task="classification",
    metric="accuracy",
    time_budget=TIME_BUDGET, # Tempo disponível para buscar os modelos.
    n_jobs=N_JOBS,
    seed=RANDOM_STATE,
)

tempo_flaml = time.perf_counter() - t0 # Tempo total do treino.
mem_flaml = psutil.Process().memory_info().rss / 1024**2 - mem_antes

y_pred_flaml = automl_flaml.predict(X_test)
acc_flaml = accuracy_score(y_test, y_pred_flaml)

# Mostra os resultados e o modelo escolhido pelo FLAML.
print(f"\nAcurácia FLAML (teste): {acc_flaml:.4f}")
print("Melhor algoritmo:", automl_flaml.best_estimator)
print("Melhor configuração:", automl_flaml.best_config)
print("Modelo final:", getattr(automl_flaml.model, "estimator", automl_flaml.model))
print("Tempo para treinar a melhor configuração (s):", getattr(automl_flaml, "best_config_train_time", "n/d"))
print("Tempo até achar o melhor modelo (s):", getattr(automl_flaml, "time_to_find_best_model", "n/d"))
print("Tentativas realizadas:", len(getattr(automl_flaml, "config_history", {})))

[flaml.automl.logger: 11-13 19:00:03] {2484} INFO - task = classification
[flaml.automl.logger: 11-13 19:00:03] {2495} INFO - Evaluation method: cv
[flaml.automl.logger: 11-13 19:00:03] {2604} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 11-13 19:00:03] {2721} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 11-13 19:00:03] {3026} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 11-13 19:00:03] {3161} INFO - Estimated sufficient time budget=1940s. Estimated necessary time budget=45s.
[flaml.automl.logger: 11-13 19:00:03] {3212} INFO -  at 0.3s,	estimator lgbm's best error=5.8333e-02,	best estimator lgbm's best error=5.8333e-02
[flaml.automl.logger: 11-13 19:00:03] {3026} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 11-13 19:00:03] {3212} INFO -  at 0.5s,	estimator lgbm's best error=5.8333e-02,	best estimator lgbm's best error=5.8333e-02
[flaml.auto

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 11-13 19:00:04] {3212} INFO -  at 1.0s,	estimator sgd's best error=5.6667e-01,	best estimator lgbm's best error=5.8333e-02
[flaml.automl.logger: 11-13 19:00:04] {3026} INFO - iteration 4, current learner lgbm
[flaml.automl.logger: 11-13 19:00:04] {3212} INFO -  at 1.3s,	estimator lgbm's best error=5.8333e-02,	best estimator lgbm's best error=5.8333e-02
[flaml.automl.logger: 11-13 19:00:04] {3026} INFO - iteration 5, current learner xgboost
[flaml.automl.logger: 11-13 19:00:04] {3212} INFO -  at 1.7s,	estimator xgboost's best error=5.0000e-02,	best estimator xgboost's best error=5.0000e-02
[flaml.automl.logger: 11-13 19:00:04] {3026} INFO - iteration 6, current learner extra_tree
[flaml.automl.logger: 11-13 19:00:05] {3212} INFO -  at 1.9s,	estimator extra_tree's best error=1.5833e-01,	best estimator xgboost's best error=5.0000e-02
[flaml.automl.logger: 11-13 19:00:05] {3026} INFO - iteration 7, current learner xgboost
[flaml.automl.logger: 11-13 19:00:05] {3212} I

## **6 - Auto-Sklearn**

O Auto-Sklearn roda em outro ambiente, mas usa os mesmos dados de treino e teste do FLAML.

### 6.1 - Salvando os dados

Neste projeto, uso Python 3.9 em um ambiente separado para rodar o Auto-Sklearn, enquanto o notebook principal usa Python 3.13.

Salvo os dados de treino e teste em CSV para passar a mesma divisão para ele. O script faz o treino e salva as previsões e os resultados para eu ler depois no notebook.

In [6]:
# Salva os mesmos dados para o Auto-Sklearn usar.
os.makedirs("askl_data", exist_ok=True)
X_train.to_csv("askl_data/X_train.csv", index=False)
X_test.to_csv("askl_data/X_test.csv", index=False)
y_train.to_frame().to_csv("askl_data/y_train.csv", index=False)
y_test.to_frame().to_csv("askl_data/y_test.csv", index=False)
print(os.listdir("askl_data"))

['X_train.csv', 'X_test.csv', 'y_test.csv', 'y_train.csv']


### 6.2 - Criando o ambiente e instalando o Auto-Sklearn

Aqui recrio o ambiente com Python 3.9 e instalo o Auto-Sklearn e as bibliotecas que ele precisa.

In [7]:
%%bash
# Recria o ambiente com Python 3.9.
rm -rf askl_env
pip install -q uv
uv python install 3.9
uv venv askl_env --python 3.9
export PATH="$PWD/askl_env/bin:$PATH"

# Instala as dependências com as versões definidas.
uv pip install -q --python askl_env/bin/python swig "setuptools<70" wheel "cython<3" "numpy<1.24" "pandas<2"
uv pip install -q --python askl_env/bin/python --no-build-isolation auto-sklearn psutil
askl_env/bin/python -c "import autosklearn; print('auto-sklearn', autosklearn.__version__, 'instalado')"

auto-sklearn 0.15.0 instalado


Installed Python 3.9.25 in 1.45s
 + cpython-3.9.25-linux-x86_64-gnu (python3.9)
Using CPython 3.9.25
Creating virtual environment at: askl_env
Activate with: source askl_env/bin/activate


### 6.3 - Script do Auto-Sklearn

O script lê os dados salvos, treina o modelo e grava as previsões e as métricas. Também mede o tempo e acompanha a memória do processo e dos subprocessos durante o treino, guardando o maior valor encontrado.

In [8]:
%%writefile askl_script.py
import json
import time
import sys
import threading
import psutil
import pandas as pd
import autosklearn.classification as ac
from sklearn.metrics import accuracy_score

class MonitorMemoria:
    # Confere a memória do processo e dos subprocessos a cada 0,2 segundo, por padrão.
    def __init__(self, intervalo=0.2):
        self.intervalo = intervalo
        self.pico_mb = 0.0
        self._parar = threading.Event()
        self._proc = psutil.Process()

    def _total_mb(self):
        total = self._proc.memory_info().rss
        total += sum(c.memory_info().rss for c in self._proc.children(recursive=True))
        return total / 1024**2

    def _loop(self):
        while not self._parar.is_set():
            try:
                self.pico_mb = max(self.pico_mb, self._total_mb())
            except psutil.Error:
                pass
            time.sleep(self.intervalo)

    def __enter__(self):
        self._t = threading.Thread(target=self._loop, daemon=True)
        self._t.start()
        return self

    def __exit__(self, *args):
        self._parar.set()
        self._t.join()

TIME_BUDGET = int(sys.argv[1])
PER_RUN_LIMIT = int(sys.argv[2])
SEED = int(sys.argv[3])

X_train = pd.read_csv("askl_data/X_train.csv")
X_test = pd.read_csv("askl_data/X_test.csv")
y_train = pd.read_csv("askl_data/y_train.csv").iloc[:, 0]
y_test = pd.read_csv("askl_data/y_test.csv").iloc[:, 0]

modelo = ac.AutoSklearnClassifier(
    time_left_for_this_task=TIME_BUDGET,
    per_run_time_limit=PER_RUN_LIMIT,
    seed=SEED,
    n_jobs=1,
    memory_limit=3072,
)

# Memória inicial do processo principal, usada como referência.
base_memoria_mb = psutil.Process().memory_info().rss / 1024**2

with MonitorMemoria() as mon:
    inicio = time.perf_counter()
    modelo.fit(X_train, y_train)
    tempo = time.perf_counter() - inicio

y_pred = modelo.predict(X_test)
pd.DataFrame({"pred": y_pred}).to_csv("askl_data/askl_pred.csv", index=False)

# Lista os modelos do ensemble, seus pesos, custos de validação e etapas.
linhas = []
for model_id, info in modelo.show_models().items():
    linhas.append(f"modelo {model_id} | peso {info['ensemble_weight']:.2f} | custo {info['cost']:.4f}")
    linhas.append(f"  pré-processador de dados   : {type(info['data_preprocessor'].choice).__name__}")
    linhas.append(f"  pré-processador de features: {type(info['feature_preprocessor'].choice).__name__}")
    linhas.append(f"  classificador              : {info['sklearn_classifier']}")
    linhas.append("")

resultado = {
    "tempo_s": tempo,
    "configuracoes": int(len(modelo.cv_results_["mean_test_score"])),
    "modelos_no_ensemble": int(len(modelo.get_models_with_weights())),
    "pico_memoria_mb": mon.pico_mb,
    "memoria_adicional_mb": max(0.0, mon.pico_mb - base_memoria_mb), # Tira a memória inicial do processo principal do pico registrado.
    "acuracia_teste": float(accuracy_score(y_test, y_pred)),
    "estatisticas": modelo.sprint_statistics(),
    "leaderboard": modelo.leaderboard(detailed=False).to_string(),
    "modelos": "\n".join(linhas),
}
with open("askl_data/askl_result.json", "w") as f:
    json.dump(resultado, f)
print("auto-sklearn finalizado.")

Writing askl_script.py


### 6.4 - Rodando o script e lendo os resultados

Apago os arquivos de resultados antigos e rodo o script. Se tudo der certo, carrego os resultados para comparar com o FLAML. Se der algum erro, mostro a mensagem que o processo retornou.

In [9]:
# Apaga os resultados antigos antes de rodar de novo.
for arq in ("askl_data/askl_result.json", "askl_data/askl_pred.csv"):
    if os.path.exists(arq):
        os.remove(arq)

AUTOSKLEARN_OK = False
proc = subprocess.run(
    ["askl_env/bin/python", "askl_script.py", str(TIME_BUDGET), str(PER_RUN_LIMIT), str(RANDOM_STATE)],
    capture_output=True, text=True
)
print(proc.stdout[-1500:])

if proc.returncode == 0 and os.path.exists("askl_data/askl_result.json"):
    AUTOSKLEARN_OK = True
    with open("askl_data/askl_result.json") as f:
        res_askl = json.load(f)

    # Carrega os resultados nas variáveis usadas na comparação.
    y_pred_ask = pd.read_csv("askl_data/askl_pred.csv")["pred"].to_numpy()
    acc_ask = accuracy_score(y_test, y_pred_ask)
    tempo_ask = res_askl["tempo_s"]
    mem_ask = res_askl["memoria_adicional_mb"]
    print(f"\nAcurácia Auto-Sklearn (teste): {acc_ask:.4f}")
    print("Tempo real de execução (s):", round(tempo_ask, 2))
else:
    print("Falha ao executar o auto-sklearn:")
    print(proc.stderr[-3000:])

auto-sklearn finalizado.


Acurácia Auto-Sklearn (teste): 0.9667
Tempo real de execução (s): 53.39


In [10]:
# Mostra as estatísticas e os modelos do Auto-Sklearn.
if AUTOSKLEARN_OK:
    print(res_askl["estatisticas"])
    print()
    print(res_askl["leaderboard"])
    print()
    print(res_askl["modelos"])
else:
    print("Auto-Sklearn indisponível neste ambiente.")

auto-sklearn results:
  Dataset name: a5c17a33-df94-11f1-8645-fb990d84e4c4
  Metric: accuracy
  Best validation score: 0.975000
  Number of target algorithm runs: 21
  Number of successful target algorithm runs: 13
  Number of crashed target algorithm runs: 8
  Number of target algorithms that exceeded the time limit: 0
  Number of target algorithms that exceeded the memory limit: 0


          rank  ensemble_weight                type   cost  duration
model_id                                                            
2            1             0.04       random_forest  0.025  4.816774
3            2             0.14       random_forest  0.025  4.354127
5            3             0.04       random_forest  0.025  3.463737
21           4             0.12         extra_trees  0.025  1.623682
9            5             0.12       random_forest  0.050  1.909487
15           6             0.04            adaboost  0.050  1.591244
20           7             0.10   gradient_boosting  0.050  

## **7 - Comparação dos resultados**

Aqui mostro a acurácia e o relatório de classificação de cada ferramenta, com precisão, recall e F1 por espécie. Depois, junto os resultados de acurácia, tempo e memória em uma tabela. Como o teste tem 30 amostras, cada erro muda a acurácia em cerca de 3,33 pontos percentuais.

In [11]:
previsoes = {"FLAML": y_pred_flaml}
if AUTOSKLEARN_OK:
    previsoes["Auto-Sklearn"] = y_pred_ask

for nome, y_pred in previsoes.items():
    print(f"\n{nome} — Acurácia no teste: {accuracy_score(y_test, y_pred):.2%}")
    print(classification_report(y_test, y_pred, zero_division=0))

# Junta os resultados na tabela final.
linhas = [{"Ferramenta": "FLAML", "Acurácia (teste)": round(acc_flaml, 4),
           "Tempo real (s)": round(tempo_flaml, 1), "Memória do processo (MB)": round(mem_flaml, 1)}]
if AUTOSKLEARN_OK:
    linhas.append({"Ferramenta": "Auto-Sklearn", "Acurácia (teste)": round(acc_ask, 4),
                   "Tempo real (s)": round(tempo_ask, 1), "Memória do processo (MB)": round(mem_ask, 1)})
tabela_final = pd.DataFrame(linhas).set_index("Ferramenta")
print()
print(tabela_final)

if AUTOSKLEARN_OK:
    if np.isclose(acc_flaml, acc_ask):
        print("\nHouve EMPATE na acurácia.")
    else:
        print("\nMelhor acurácia:", "FLAML" if acc_flaml > acc_ask else "Auto-Sklearn")
print(f"O teste tem {len(y_test)} amostras, então cada erro vale {100 / len(y_test):.2f} p.p.")


FLAML — Acurácia no teste: 93.33%
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       0.90      0.90      0.90        10
 Iris-virginica       0.90      0.90      0.90        10

       accuracy                           0.93        30
      macro avg       0.93      0.93      0.93        30
   weighted avg       0.93      0.93      0.93        30


Auto-Sklearn — Acurácia no teste: 96.67%
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.90      0.95        10
 Iris-virginica       0.91      1.00      0.95        10

       accuracy                           0.97        30
      macro avg       0.97      0.97      0.97        30
   weighted avg       0.97      0.97      0.97        30


              Acurácia (teste)  Tempo real (s)  Memória do processo (MB)
Ferramenta                                   

## **8 - Relatório**

Os comentários abaixo são sobre os resultados salvos neste notebook. Se rodar de novo, os modelos e os resultados podem mudar.

**Desempenho:** Usei os mesmos dados nas duas ferramentas: 120 amostras para treino e 30 para teste, com 60 segundos de busca e semente 42. O FLAML acertou 26 amostras e ficou com 86,67% de acurácia. O Auto-Sklearn acertou 29 e chegou a 96,67%, errando uma versicolor como virginica. Então, nessa execução, o Auto-Sklearn ficou 10 pontos percentuais à frente. Mesmo assim, o teste é pequeno e só foi feita uma divisão dos dados. Para saber se essa diferença se mantém, seria preciso testar outras sementes ou fazer uma comparação com validação cruzada.

**Tempo e memória:** O FLAML levou cerca de 60,0 segundos para treinar, e o Auto-Sklearn, 58,7 segundos. Na memória, os valores registrados foram 10,4 MB para o FLAML e 272,2 MB para o Auto-Sklearn. Só que o código mede isso de formas diferentes: no FLAML, compara a memória do processo antes e depois do treino; no Auto-Sklearn, acompanha o pico do processo e dos subprocessos e desconta a memória inicial do processo principal. Por isso, não dá para comparar esses números diretamente. O Auto-Sklearn também precisou de um ambiente separado com Python 3.9, e o tempo dessa instalação não entra no tempo de treino. Das 28 tentativas dele, 17 deram certo, 10 falharam e uma passou do limite de tempo.

**Modelos escolhidos:** O FLAML escolheu um `ExtraTreesClassifier` com 22 árvores, limite de 9 nós folha por árvore e critério de entropia. Já o Auto-Sklearn juntou 15 modelos em um ensemble, ou seja, uma combinação de modelos com pesos diferentes. Entre eles estão florestas aleatórias, Extra Trees, AdaBoost, boosting, SVM, redes neurais, árvore de decisão e classificador passivo-agressivo. Eles também usam diferentes etapas de preparação dos dados, o que deixa a solução mais trabalhosa de entender. O modelo do FLAML também combina árvores, mas tem menos componentes para analisar. Nessa execução, o Auto-Sklearn acertou mais no teste. Ainda assim, além dos acertos, é preciso considerar o trabalho para preparar o ambiente, a complexidade dos modelos e as diferenças na forma de medir a memória.